<a href="https://colab.research.google.com/github/hamza-24-ai/Pytorch_with_practical_deepLearning/blob/main/08_CNN_CIFAR10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim
from torchvision import datasets, transforms

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "CPU")
print(f"device is running on {device}")

device is running on cuda


In [3]:
torch.manual_seed(42)

In [5]:
transform = transforms.Compose([
    transforms.ToTensor(),

    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

print("Ready to convert image into tensor of pixels")

Ready to convert image into tensor of pixels


In [6]:
# Load and download dataset from datasets

train_dataset = datasets.CIFAR10("./data", train=True,download=True, transform=transform)
test_dataset = datasets.CIFAR10("./data", train=True,download=True, transform=transform)

print("data has been trained and download successfully")

100%|██████████| 170M/170M [39:44<00:00, 71.5kB/s]


data has been trained and download successfully


In [7]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

print("Data Loader Successfully ")

Data Loader Successfully 


In [17]:
class MyNN(nn.Module):
  def __init__(self,input_features):
    super().__init__()

    self.features = nn.Sequential(
        nn.Conv2d(input_features,32,kernel_size=3,padding="same"),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(kernel_size=2, stride=2),


        nn.Conv2d(32,64,kernel_size=3,padding="same"),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )

    self.sequence = nn.Sequential(
        nn.Flatten(),
        nn.Linear(64*8*8,128),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(64,10)
    )

  def forward(self,x):

    x = self.features(x)
    x = self.sequence(x)

    return x

In [18]:
learning_rate = 0.001
epochs = 20

In [19]:
model = MyNN(3)
model.to(device)

criterain = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = learning_rate, weight_decay=1e-4)


In [20]:
for epoch in range(epochs):

  model.train()
  total_epoch_loss = 0

  for batch_features, batch_labels in train_loader:

    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)
    loss = criterain(outputs, batch_labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_epoch_loss += loss.item()

  avg_loss = total_epoch_loss/len(train_loader)

  print(f"Epoch {epoch + 1} loss has : {avg_loss:.4f}")




Epoch 1 loss has : 1.4621
Epoch 2 loss has : 1.1165
Epoch 3 loss has : 0.9720
Epoch 4 loss has : 0.8760
Epoch 5 loss has : 0.7982
Epoch 6 loss has : 0.7330
Epoch 7 loss has : 0.6852
Epoch 8 loss has : 0.6409
Epoch 9 loss has : 0.6032
Epoch 10 loss has : 0.5765
Epoch 11 loss has : 0.5417
Epoch 12 loss has : 0.5195
Epoch 13 loss has : 0.4926
Epoch 14 loss has : 0.4822
Epoch 15 loss has : 0.4612
Epoch 16 loss has : 0.4500
Epoch 17 loss has : 0.4350
Epoch 18 loss has : 0.4255
Epoch 19 loss has : 0.4124
Epoch 20 loss has : 0.3970


In [21]:
model.eval()

MyNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (sequence): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [23]:
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:

    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _,prediction = torch.max(outputs,1)

    total += batch_labels.size(0)
    correct += (prediction == batch_labels).sum().item()

print(f"Accuracy is : {correct/total * 100}:.4f")


Accuracy is : 95.006:.4f
